# Understat vs FPL comparison

Checks the Understat xG/xA data merged into `data/merged_data.csv` (see `get_understat_data.py` and `src/understat.py`):

1. Per-season distributions of Understat's attacking stats.
2. Understat vs FPL's own native `expected_goals`/`expected_assists`, where both exist.

FPL only started publishing its own expected stats from **2022-23** onwards (2021-22 is entirely null), so the comparison in part 2 is only possible for 2022-23 through 2024-25. 2021-22 is the season Understat actually backfills for us — there's nothing to compare it against.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import pearsonr

from src.constants import UNDERSTAT_SEASONS

PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

df = pd.read_csv("data/merged_data.csv", low_memory=False)
df = df[df["minutes"] > 0].copy()  # attacking-output distributions only make sense while playing

print(f"Player-gameweek rows with minutes > 0: {len(df)}")
print(f"Seasons: {sorted(df['season'].unique())}")
print(f"Understat-covered seasons: {UNDERSTAT_SEASONS}")

## Per-season Understat distributions

Summary stats and boxplots for `understat_xG`/`understat_xA`, restricted to player-gameweeks with minutes > 0 and to seasons Understat actually covers.

In [ ]:
UNDERSTAT_METRICS = ["understat_xG", "understat_xA", "understat_npxG", "understat_shots", "understat_key_passes"]

understat_df = df[df["season"].isin(UNDERSTAT_SEASONS)]

summary = understat_df.groupby("season")[UNDERSTAT_METRICS].agg(["mean", "median", "std"])
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric in zip(axes, ["understat_xG", "understat_xA"]):
    data = [understat_df.loc[understat_df["season"] == s, metric] for s in UNDERSTAT_SEASONS]
    ax.boxplot(data, labels=UNDERSTAT_SEASONS, showfliers=False)
    ax.set_title(f"{metric} by season (minutes > 0)")
    ax.set_ylabel(metric)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
zero_share = understat_df.groupby("season")["understat_xG"].apply(lambda s: (s == 0).mean())

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(zero_share.index, zero_share.values, color=PALETTE[0])
ax.set_ylabel("Share of player-gameweeks with understat_xG == 0")
ax.set_title("Zero-xG share by season (minutes > 0)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Understat vs FPL native expected stats

Only seasons where FPL's own `expected_goals`/`expected_assists` are populated (2022-23 onwards) can be compared.

In [ ]:
OVERLAP_SEASONS = [s for s in UNDERSTAT_SEASONS if df.loc[df["season"] == s, "expected_goals"].notna().any()]
print(f"Seasons with both Understat and FPL native expected stats: {OVERLAP_SEASONS}")

comparison = df[df["season"].isin(OVERLAP_SEASONS)].dropna(
    subset=["expected_goals", "expected_assists", "understat_xG", "understat_xA"]
)
print(f"Player-gameweek rows in the comparison: {len(comparison)}")

In [ ]:
def plot_comparison_by_season(data: pd.DataFrame, understat_col: str, fpl_col: str, label: str) -> None:
    """Per-season scatter of Understat vs FPL native expected stats, with r and MAE annotated."""
    fig, axes = plt.subplots(1, len(OVERLAP_SEASONS), figsize=(5 * len(OVERLAP_SEASONS), 5), sharex=True, sharey=True)

    for ax, season, color in zip(axes, OVERLAP_SEASONS, PALETTE):
        s = data[data["season"] == season]
        r, _ = pearsonr(s[understat_col], s[fpl_col])
        mae = (s[understat_col] - s[fpl_col]).abs().mean()

        ax.scatter(s[understat_col], s[fpl_col], alpha=0.15, s=10, color=color)
        max_val = max(s[understat_col].max(), s[fpl_col].max())
        ax.plot([0, max_val], [0, max_val], "r--", linewidth=1, label="Perfect agreement")
        ax.set_title(f"{season}\nr={r:.3f}, MAE={mae:.3f}, n={len(s)}")
        ax.set_xlabel(f"Understat {label}")
        ax.grid(alpha=0.3)

    axes[0].set_ylabel(f"FPL {label}")
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_comparison_by_season(comparison, "understat_xG", "expected_goals", "xG")

In [ ]:
plot_comparison_by_season(comparison, "understat_xA", "expected_assists", "xA")

### Player-season totals

Per-gameweek agreement is noisy (single-match variance, rounding). Summing to a player-season total is a cleaner check of whether the two sources broadly agree.

In [ ]:
season_totals = (
    comparison.groupby(["season", "element", "name"])[
        ["understat_xG", "expected_goals", "understat_xA", "expected_assists"]
    ]
    .sum()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (understat_col, fpl_col, label) in zip(
    axes, [("understat_xG", "expected_goals", "xG"), ("understat_xA", "expected_assists", "xA")]
):
    r, _ = pearsonr(season_totals[understat_col], season_totals[fpl_col])
    ax.scatter(season_totals[understat_col], season_totals[fpl_col], alpha=0.4, s=15, color=PALETTE[0])
    max_val = max(season_totals[understat_col].max(), season_totals[fpl_col].max())
    ax.plot([0, max_val], [0, max_val], "r--", linewidth=1)
    ax.set_xlabel(f"Understat {label} (season total)")
    ax.set_ylabel(f"FPL {label} (season total)")
    ax.set_title(f"Season-total {label}: r={r:.3f}")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
season_totals["xG_diff"] = season_totals["understat_xG"] - season_totals["expected_goals"]

print("Biggest season-total xG disagreements (Understat minus FPL):")
season_totals.assign(abs_diff=season_totals["xG_diff"].abs()) \
    .sort_values("abs_diff", ascending=False) \
    .head(10)[["season", "name", "understat_xG", "expected_goals", "xG_diff"]]

## Caveats

- **Double gameweeks**: `src/clean.py` collapses a double-gameweek player to a single row (the fixture with the most minutes), discarding the other fixture's points entirely. `src/understat.py` sums Understat stats across *all* fixtures in a gameweek, so for double-gameweek rows the Understat total can reflect two matches while the FPL columns on that same row only reflect one. Pre-existing limitation of the dedup step, not something this merge introduces.
- **Player coverage**: not every Understat player resolves to an FPL `code` (some rows are non-PL competitions from a player's full career history that share a season label by coincidence; a handful are genuine name-matching misses for 2023-24/2024-25, which lack an `id_dict.csv`). Coverage checked at merge time was 90-102% of FPL players with minutes each season — see `src/understat.py`.
- **2021-22**: FPL has no native `expected_goals`/`expected_assists` at all this season, so there's no comparison possible — Understat is the only source, which is the whole point of pulling it in.